# H17 (Round 9) - cross-lingual manifold retrain

**Author**: kj  
**Approach**: retrain the frozen 18-feature HIGH manifold on gold v2 (first gold with non-English negatives) and test whether it catches non-English hallucinations.

The shipped manifold catches 0 of 139 non-English hallucinations (TNR 0.000) while English is healthy (TNR 0.710). A probe showed the features already separate the non-English classes (r1_mt AUC 0.80), so the defect is the weights, not the features. This notebook drives `experiments/grounding/round9.py` against the cached feature table and records the honest held-out result (5-fold OOF + leave-one-language-out).

**Finding**: retrain-alone misses the 0.30 ship bar (OOF non-EN TNR 0.13) but *improves* English (TNR 0.710 -> 0.850). The miss is an operating-point problem - a language-conditional non-EN threshold (~0.65) reaches LOLO non-EN TNR 0.60-0.93 at TPR 0.50-0.87, generalizing to unseen languages. Shipped weights/config untouched (retrain writes an experiment copy).

## Imports

In [1]:
import sys
from pathlib import Path

# round9.py lives in experiments/grounding and uses __file__-relative data paths (cwd-independent)
GROUNDING = Path.cwd().parent / "experiments" / "grounding"
sys.path.insert(0, str(GROUNDING))

import round9 as R  # noqa: E402

## Stage 1 - do the shipped features separate the non-English classes?

Per-feature AUC for catching a non-English hallucination (1.0 = perfect ranking), MT firing fraction, and per base-language `r1_best` AUC. Builds the feature cache on first run (~3 min), reuses it after.

In [2]:
R.cmd_audit()

non-EN rows 1343  neg 139  pos 1204

--- per-feature AUC for catching non-EN hallucination (1.0=perfect) ---
  r1_direct          AUC=0.592  supp=0.241  halluc=0.158
  r1_mt              AUC=0.806  supp=0.528  halluc=0.245
  r1_best            AUC=0.803  supp=0.537  halluc=0.262
  charng             AUC=0.583  supp=0.269  halluc=0.190
  fuzzy              AUC=0.646  supp=0.501  halluc=0.438
  anchor             AUC=0.511  supp=0.457  halluc=0.439
  anchor_mm          AUC=0.416  supp=0.163  halluc=0.331
  oracle             AUC=0.583  supp=0.247  halluc=0.168
  top3               AUC=0.638  supp=0.233  halluc=0.130
  same_lang          AUC=0.472  supp=0.209  halluc=0.266
  is_en              AUC=0.526  supp=0.052  halluc=0.000
  specificity        AUC=0.504  supp=0.148  halluc=0.123
  conflict_n         AUC=0.505  supp=0.005  halluc=0.000
  conflict_flag      AUC=0.505  supp=0.011  halluc=0.000
  num_edit_mag       AUC=0.505  supp=0.008  halluc=0.000
  wn_antonym_flip    AUC=0.501  supp

## Stage 3 - baseline vs retrained, held-out (5-fold OOF) + LOLO

Slice metrics: TNR = hallucination recall (caught / total negatives), TPR = support confirm rate, balanced-acc = (TNR+TPR)/2. The retrain uses a single global threshold here - LOLO at that threshold collapses for fr/nb, the symptom the next cell diagnoses.

In [3]:
R.cmd_eval()

/home/lab/workspace/private/ai-assistants/claude-code-plugins/experiments/grounding/round9.py:150: RuntimeWarning: Mean of empty slice
  ba = np.nanmean([tnr, tpr])



=== BASELINE shipped HIGH (gold v2) ===
  non_EN   n= 1343 neg= 139  TNR=0.000  TPR=0.997  bal_acc=0.498
  EN       n= 4569 neg=1826  TNR=0.710  TPR=0.884  bal_acc=0.797
  vitaminc n=    0 neg=   0  TNR=nan  TPR=nan  bal_acc=nan


/home/lab/workspace/private/ai-assistants/claude-code-plugins/experiments/grounding/round9.py:150: RuntimeWarning: Mean of empty slice
  ba = np.nanmean([tnr, tpr])



=== RETRAINED gold v2 - 5-fold OOF (oversample_ne=1) ===
  non_EN   n= 1343 neg= 139  TNR=0.122  TPR=0.974  bal_acc=0.548
  EN       n= 4569 neg=1826  TNR=0.827  TPR=0.818  bal_acc=0.823
  vitaminc n=    0 neg=   0  TNR=nan  TPR=nan  bal_acc=nan


/home/lab/workspace/private/ai-assistants/claude-code-plugins/experiments/grounding/round9.py:150: RuntimeWarning: Mean of empty slice
  ba = np.nanmean([tnr, tpr])



=== RETRAINED gold v2 - 5-fold OOF (oversample_ne=3) ===
  non_EN   n= 1343 neg= 139  TNR=0.129  TPR=0.967  bal_acc=0.548
  EN       n= 4569 neg=1826  TNR=0.850  TPR=0.793  bal_acc=0.821
  vitaminc n=    0 neg=   0  TNR=nan  TPR=nan  bal_acc=nan

=== LOLO (leave-one-language-out) held-out non-EN TNR, oversample_ne=3 ===
  es: held-out neg= 35  TNR=0.200  (confirm rate on its 212 pos=0.972)


  fr: held-out neg= 28  TNR=0.000  (confirm rate on its 422 pos=0.991)


  pt: held-out neg= 20  TNR=0.100  (confirm rate on its 37 pos=0.946)
  nb: held-out neg= 17  TNR=0.000  (confirm rate on its 254 pos=0.992)


  sv: held-out neg= 14  TNR=0.214  (confirm rate on its 18 pos=0.944)


## Operating-point diagnosis - a language-conditional threshold

The non-English miss is the global threshold (calibrated to the English bulk), not the signal. Sweep a non-English-specific threshold on the OOF probabilities (English keeps its own), then confirm it survives LOLO on a held-out language never seen in training.

In [4]:
R.cmd_threshold()

=== non-EN threshold sweep (OOF probabilities, 1204 pos / 139 neg) ===
  thr   TNR(catch)  TPR(confirm)  bal_acc
  0.30   0.065       0.989        0.527 <- best @TPR>=0.90
  0.35   0.144       0.978        0.561 <- best @TPR>=0.90
  0.40   0.216       0.960        0.588 <- best @TPR>=0.90
  0.45   0.295       0.924        0.610 <- best @TPR>=0.90
  0.50   0.396       0.903        0.649 <- best @TPR>=0.90
  0.55   0.482       0.875        0.679
  0.60   0.590       0.842        0.716
  0.65   0.676       0.797        0.736
  0.70   0.748       0.758        0.753
  0.75   0.813       0.711        0.762
  0.80   0.863       0.659        0.761
  0.85   0.878       0.593        0.735
  0.90   0.921       0.512        0.717
  0.95   0.950       0.367        0.658

Reachable @TPR>=0.90: TNR=0.396 at non-EN thr=0.50 (shipped single thr catches 0.000)

=== LOLO at fixed non-EN thresholds (held-out language never in training) ===
  non-EN threshold = 0.65
    es: held neg= 35  TNR=0.743  TPR=0.6

    fr: held neg= 28  TNR=0.643  TPR=0.865
    pt: held neg= 20  TNR=0.600  TPR=0.784


    nb: held neg= 17  TNR=0.647  TPR=0.843
    sv: held neg= 14  TNR=0.929  TPR=0.500
  non-EN threshold = 0.7


    es: held neg= 35  TNR=0.771  TPR=0.613
    fr: held neg= 28  TNR=0.714  TPR=0.834


    pt: held neg= 20  TNR=0.600  TPR=0.784
    nb: held neg= 17  TNR=0.647  TPR=0.799


    sv: held neg= 14  TNR=0.929  TPR=0.500


## Conclusion

- Features separate the non-English classes (r1_mt AUC 0.806); the retrain weights r1_mt +2.36 (shipped ~0, killed by English collinearity)
- Retrain alone: OOF non-EN TNR 0.000 -> 0.13 (misses the 0.30 bar), English TNR 0.710 -> 0.850 (improved)
- A non-English threshold ~0.65 reaches OOF TNR 0.68 / TPR 0.80 and generalizes - LOLO held-out TNR 0.60-0.93
- The fix is deterministic and in-contract: retrained weights + a language-conditional threshold keyed off `is_en` (already computed). Shipping needs a `LexicalVerdict.confirmed` + config change - held for approval
- Shipped weights/config untouched; retrain writes `config_document_processing.experiment.yaml` (gitignored)